# Model Deployment Notebook

> 📊 **Best viewed on [nbviewer](https://nbviewer.org/github/aws-samples/sample-mlops-bestpractices/blob/main/sagemaker-automated-drift-and-trend-monitoring/notebooks/2_deployment.ipynb)** — GitHub's renderer strips JavaScript that powers interactive output cells (Evidently reports, plotly charts, ipywidgets). nbviewer (run by Project Jupyter) renders them in full.


Deploy the trained fraud detection model to a SageMaker serverless endpoint.

This notebook handles:
1. **Model Selection** — Choose from trained models in the Model Registry
2. **Endpoint Creation** — Deploy to a serverless endpoint with custom inference handler
3. **Endpoint Testing** — Verify the endpoint works correctly
4. **Endpoint Management** — Update or delete endpoints

**Prerequisites:**
- Run `1_training_pipeline.ipynb` first to train a model
- Model must be registered in SageMaker Model Registry

**Custom Inference Handler:**
The endpoint uses a custom handler that automatically logs all predictions to Athena for drift monitoring.

**Environment:** SageMaker AI Notebook or local with AWS credentials.

## Setup

In [ ]:
import os
import sys
import json
import boto3
from pathlib import Path
from dotenv import load_dotenv

# Find project root and add to path
project_root = Path.cwd()
while not (project_root / '.env').exists() and project_root != project_root.parent:
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

load_dotenv(project_root / '.env')

from src.config.config import (
    AWS_DEFAULT_REGION,
    SAGEMAKER_EXEC_ROLE,
    DATA_S3_BUCKET,
    ATHENA_DATABASE,
    ATHENA_OUTPUT_S3,
    MLFLOW_MODEL_NAME,
)
from src.utils.aws_utils import get_execution_role

# Deployment logic itself lives in src/train_pipeline/deploy_endpoint.py
# (also used by `main.py deploy`) — this notebook just calls into it so
# there's a single source of truth for the deployment steps.
from src.train_pipeline.deploy_endpoint import (
    resolve_inference_sqs_queue_url,
    build_inference_env,
    select_latest_approved_model,
    create_model_from_package,
    create_endpoint_config,
    deploy_endpoint,
    get_endpoint_status,
    delete_endpoint,
)

# Initialize clients
region = AWS_DEFAULT_REGION
sm_client = boto3.client('sagemaker', region_name=region)
runtime_client = boto3.client('sagemaker-runtime', region_name=region)
s3_client = boto3.client('s3', region_name=region)
sqs_client = boto3.client('sqs', region_name=region)

role = get_execution_role()

print(f'Region: {region}')
print(f'Role: {role}')
print(f'S3 Bucket: {DATA_S3_BUCKET}')
print(f'Athena DB: {ATHENA_DATABASE}')
print(f'MLflow Model: {MLFLOW_MODEL_NAME}')

## Deployment Configuration

In [ ]:
# Endpoint configuration (overrides passed to the deploy_endpoint.py functions below).
ENDPOINT_NAME = 'fraud-detector-endpoint'
# MODEL_PACKAGE_GROUP matches the MLflow registered model name (from config.yaml).
# Pipeline.py registers each trained model into a SageMaker Model Package Group
# of the same name (pipeline.py:321), so the two registries stay in lockstep.
MODEL_PACKAGE_GROUP = MLFLOW_MODEL_NAME

# Serverless configuration
MEMORY_SIZE_MB = 2048  # 1024, 2048, 3072, 4096, 5120, 6144
MAX_CONCURRENCY = 10   # Max concurrent invocations

# Resolve the inference-logging SQS queue URL and build the custom inference
# handler's environment. CFN creates the queue (resource `InferenceLoggerQueue`,
# name `${ProjectName}-inference-logging`); the handler reads SQS_QUEUE_URL
# from its env and sends a message per prediction, and a downstream Lambda
# drains the queue into Athena `inference_responses`. If the queue can't be
# resolved, the endpoint still deploys but Athena logging is disabled.
INFERENCE_SQS_QUEUE_URL = resolve_inference_sqs_queue_url(sqs_client=sqs_client)
INFERENCE_ENV = build_inference_env(ENDPOINT_NAME, INFERENCE_SQS_QUEUE_URL, region)

print('Endpoint Configuration:')
print(f'  Name: {ENDPOINT_NAME}')
print(f'  Memory: {MEMORY_SIZE_MB} MB')
print(f'  Max Concurrency: {MAX_CONCURRENCY}')
print(f'\nAthena Logging: {INFERENCE_ENV["ENABLE_ATHENA_LOGGING"]}')
print(f'SQS Queue: {INFERENCE_SQS_QUEUE_URL or "(unresolved — logging disabled)"}')

## 1. Select Model from Registry

Choose which model version to deploy.

In [ ]:
# Select the latest approved model from the registry (set model_version=N to pin a specific one).
print(f'Available models in package group: {MODEL_PACKAGE_GROUP}\n')

model_info = select_latest_approved_model(MODEL_PACKAGE_GROUP, sm_client=sm_client)

LATEST_MODEL_ARN = model_info['arn']
LATEST_MODEL_VERSION = model_info['version']

print(f'✓ Will deploy model version {LATEST_MODEL_VERSION}')
print(f'  ARN: {LATEST_MODEL_ARN}')
print(f'  Created: {model_info["created_time"]}')

## 2. Create SageMaker Model with Inference Code

Package the model artifact with the custom inference handler.

In [ ]:
# Creates the SageMaker Model FROM THE MODEL PACKAGE (not from raw Image+ModelDataUrl),
# so it carries ModelPackageName — needed for the drift Lambda's
# `describe_model → ModelPackageName` lookup to succeed (no fallback to
# `list_model_packages(SortOrder=Descending)`, and no risk of comparing current
# traffic to a baseline.json belonging to a different version). Also validates
# the artifact tarball contains code/inference.py and finalizes INFERENCE_ENV
# with the model version and script-mode vars (mutated in place).
model_name = create_model_from_package(
    LATEST_MODEL_ARN,
    LATEST_MODEL_VERSION,
    ENDPOINT_NAME,
    role,
    region,
    INFERENCE_ENV,
    sm_client=sm_client,
    s3_client=s3_client,
)

print(f'✓ Model created: {model_name}')
print(f'  ModelPackage: {LATEST_MODEL_ARN}')
print(f'  Version: {INFERENCE_ENV["MODEL_VERSION"]}')

## 3. Create Endpoint Configuration

In [ ]:
ENDPOINT_CONFIG_NAME = create_endpoint_config(
    ENDPOINT_NAME,
    model_name,
    memory_size_mb=MEMORY_SIZE_MB,
    max_concurrency=MAX_CONCURRENCY,
    sm_client=sm_client,
)

print(f'✓ Endpoint configuration created: {ENDPOINT_CONFIG_NAME}')

## 4. Deploy Endpoint

Create or update the endpoint. This takes 5-10 minutes for serverless endpoints.

In [ ]:
# Deploy the endpoint (idempotent). REDEPLOY_CLEAN=True deletes any existing
# endpoint first for a clean slate; set False to update the existing
# endpoint's config in place instead. This polls until the endpoint reaches
# InService/Failed, which takes 5-10 minutes for serverless endpoints —
# progress is logged as it goes.
REDEPLOY_CLEAN = True

result = deploy_endpoint(
    ENDPOINT_NAME,
    ENDPOINT_CONFIG_NAME,
    redeploy_clean=REDEPLOY_CLEAN,
    sm_client=sm_client,
)

print(f'\n✓ Endpoint status: {result["status"]}')
print(f'  Name: {result["endpoint_name"]}')
print(f'  ARN: {result["endpoint_arn"]}')

## 5. Test Endpoint

Send a test prediction to verify the endpoint works.

In [ ]:
# Test payload (30 features)
test_features = {
    'transaction_hour': 14,
    'transaction_day_of_week': 3,
    'transaction_amount': 125.50,
    'transaction_type_code': 1,
    'customer_age': 35,
    'customer_gender': 1,
    'customer_tenure_months': 24,
    'account_age_days': 730,
    'distance_from_home_km': 5.2,
    'distance_from_last_transaction_km': 2.1,
    'time_since_last_transaction_min': 120,
    'online_transaction': 1,
    'international_transaction': 0,
    'high_risk_country': 0,
    'merchant_category_code': 5411,
    'merchant_reputation_score': 0.85,
    'chip_transaction': 1,
    'pin_used': 1,
    'card_present': 1,
    'cvv_match': 1,
    'address_verification_match': 1,
    'num_transactions_24h': 2,
    'num_transactions_7days': 8,
    'avg_transaction_amount_30days': 95.30,
    'max_transaction_amount_30days': 250.00,
    'velocity_score': 0.3,
    'recurring_transaction': 0,
    'previous_fraud_incidents': 0,
    'credit_limit': 5000.0,
    'available_credit_ratio': 0.75
}

print('Sending test prediction request...')
print(f'Test features: {json.dumps(test_features, indent=2)}\n')

try:
    response = runtime_client.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType='application/json',
        Body=json.dumps(test_features)
    )
    
    result = json.loads(response['Body'].read().decode())
    
    print('✓ Prediction successful!')
    print(f'\nResult: {json.dumps(result, indent=2)}')
    
    if 'prediction' in result:
        pred = result['prediction']
        print(f'\nFraud Detection Result:')
        print(f'  Prediction: {"FRAUD" if pred == 1 else "NOT FRAUD"}')
        if 'probability' in result:
            print(f'  Probability: {result["probability"]:.4f}')
    
except Exception as e:
    print(f'❌ Prediction failed: {e}')

## 6. Endpoint Management

Utility functions for managing the endpoint.

In [ ]:
# get_endpoint_status() and delete_endpoint() are imported from
# src/train_pipeline/deploy_endpoint.py (see Setup cell above) — no local
# wrapper definitions needed.

print('Management functions available:')
print('  - get_endpoint_status(endpoint_name, sm_client=sm_client)')
print('  - delete_endpoint(endpoint_name, sm_client=sm_client)')
print(f'\nExample: get_endpoint_status("{ENDPOINT_NAME}", sm_client=sm_client)')

## Next Steps

Now that your endpoint is deployed:

1. **Run Inference** → Go to `3_inference_monitoring.ipynb` to:
   - Send predictions to the endpoint
   - Monitor inference performance
   - Detect data and model drift

2. **View Metrics** → Go to `4_governance_dashboard.ipynb` to:
   - View model performance over time
   - Analyze drift patterns
   - Generate governance reports

3. **Monitor Endpoint** → AWS Console:
   - CloudWatch metrics for invocations and latency
   - CloudWatch logs for detailed request/response logging
   - Athena for querying logged predictions